In [ ]:
https://stats.baseboll-softboll.se/api/v1/stats/events/2025-regionserien-baseboll/index?section=teams&stats-section=batting&team=&round=&split=&split=&language=en

In [ ]:
import pandas as pd
import requests

url = "https://stats.baseboll-softboll.se/api/v1/stats/events/2025-regionserien-baseboll/index?section=teams&stats-section=batting&team=&round=&split=&split=&language=en"

headers_api = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'Accept': 'application/json'
}

try:
    response = requests.get(url, headers=headers_api)
    json_response = response.json()
    
    # --- Key fix: Enter inside data ---
    # Use pd.json_normalize to handle nested structure
    df_stats = pd.json_normalize(json_response['data'], sep='_')
    
    # Print the list of fields that truly belong to player data
    print("Actual fields of internal data:")
    print(df_stats.columns.tolist())
    
    # Based on WBSC system 2025 naming rules, try to filter correct fields
    # Usually player name is in 'player_name' or 'player_full_name'
    # Batting average may be in 'avg' or 'stats_avg'
    
    # Automatically find column names containing keywords
    def find_actual_col(keyword):
        matches = [c for c in df_stats.columns if keyword.lower() in c.lower()]
        return matches[0] if matches else None

    target_cols = {
        'player': find_actual_col('player'),
        'team': find_actual_col('team'),
        'avg': find_actual_col('avg'),
        'ops': find_actual_col('ops')
    }
    
    # Remove unfound columns
    final_cols = [v for v in target_cols.values() if v is not None]
    
    print("\nFound corresponding fields:", target_cols)
    print("\nData preview:")
    print(df_stats[final_cols].head())

except Exception as e:
    print(f"Parsing failed: {e}")

內部數據的實際欄位：
['g', 'gs', 'ab', 'r', 'h', 'double', 'triple', 'hr', 'rbi', 'tb', 'avg', 'slg', 'obp', 'ops', 'bb', 'hbp', 'so', 'gdp', 'sf', 'sh', 'sb', 'cs', 'link', 'name', 'teamid']

找到的對應欄位： {'player': None, 'team': 'teamid', 'avg': 'avg', 'ops': 'ops'}

數據預覽：
   teamid  avg  ops
0   36162  342  882
1   36171  324  922
2   36165  362  932
3   36172  311  873
4   36170  363  979
